In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
import hdbscan
import umap

In [ ]:
# reading in the words.txt
with open('../data/words.txt', 'r') as f:
    words = f.read().splitlines()

# convert to numpy array
words = np.array(words)
words

In [ ]:
len(words)

In [ ]:
# encode vectors
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(words, show_progress_bar=True)

In [ ]:
# checking if embeddings are nan
np.any(np.isnan(embeddings))

In [ ]:
# normalize the embeddings
embeddings = normalize(embeddings)

In [ ]:
# run HDBSCAN
clusterer = hdbscan.HDBSCAN(min_cluster_size=4, min_samples=4, metric='euclidean')
cluster_labels = clusterer.fit_predict(embeddings)

In [ ]:
from collections import defaultdict

clusters = defaultdict(list)

for i in range(len(words)):
    label = cluster_labels[i]
    # include the noise cluster
    clusters[label].append(words[i])

In [ ]:
len(clusters)

In [ ]:
# visualize with umap
reducer = umap.UMAP(n_neighbors=10, min_dist=0.05, metric='euclidean')
embedding_2d = reducer.fit_transform(embeddings)
plt.figure(figsize=(10, 10))
scatter = plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], c=cluster_labels, cmap='Spectral', s=50)
handles, labels = scatter.legend_elements()
plt.legend(handles, labels, title="Clusters")
plt.title('UMAP projection of the word embeddings')
plt.show()

In [ ]:
from wordcloud import WordCloud

probs = clusterer.probabilities_

# only first 5 clusters
# ignore first noise cluster
cluster_ids = list(clusters.keys())[1:5]

# for each cluster id, make the word cloud
for cid in cluster_ids:
    freqs = {}

    for i in range(len(words)):
        if cluster_labels[i] != cid:
            continue
            # weighing based on freq
        freqs[words[i]] = probs[i]

    wc = WordCloud(width=600, height=400, background_color="white")
    wc = wc.generate_from_frequencies(freqs)

    plt.figure()
    plt.imshow(wc)
    plt.axis("off")
    plt.title(f"Cluster {cid}")
    plt.show()

In [ ]:
import pandas as pd

cluster_df = pd.DataFrame({
    'cluster_id': list(clusters.keys()),
    'words': list(clusters.values())
})
cluster_df.to_pickle("../data/cluster.pkl")

In [ ]:
clusters

In [ ]:
# save embeddings to a pickle
embeddings_df = pd.DataFrame(embeddings)
embeddings_df['word'] = words
embeddings_df.to_pickle("../data/embeddings.pkl")

In [ ]:
embeddings_df.shape